# Ejercicios UD03_03

# Análisis de sentimientos con Distilbert

En la práctica [Analisis de sentimientos con DistilBERT](https://colab.research.google.com/github/martinezpenya/MIA-IABD-2425/blob/main/UD03/notebooks/3.-models_llenguatge_ES.ipynb) hemos entrenado a Distilbert para clasificar los sentimientos en los textos, utilizando una base de datos de Tweets.

En este ejercicio, queremos usar Distilbert para clasificar las reseñas de películas y el mismo entrenamiento no nos sirve. Por esta razón, utilizaremos el aprendizaje de transferencia para adaptar el modelo a la nueva tarea, utilizando el `DataSet` de las reseñas de películas IMDB, disponibles en Huggingface como un `imdb`.

## Objetivos de la práctica

* Reproduce el proceso visto en la práctica [Analisis de sentimientos con DistilBERT](https://colab.research.google.com/github/martinezpenya/MIA-IABD-2425/blob/main/UD03/notebooks/3.-models_llenguatge_ES.ipynb) para entrenar a Distilbert para clasificar los sentimientos en las reseñas de películas.
* Deberá entrenar el modelo (Distilbert u otros transformadores) con el `dataset` de las reseñas de películas IMDB.
* Deberá analizar las predicciones del modelo y compararlo con el modelo entrenado en la práctica [Analisis de sentimientos con DistilBERT](https://colab.research.google.com/github/martinezpenya/MIA-IABD-2425/blob/main/UD03/notebooks/3.-models_llenguatge_ES.ipynb).
* Utilitza un `pipeline` de transformers para hacer predicciones [`zero-shot`](https://huggingface.co/transformers/main_classes/pipelines.html#transformers.ZeroShotClassificationPipeline) sobre reseñas de películas. Comparar los resultados con los del modelo entrenado con el `dataset` de IMDB.
* Escribe una celda con tus comentarios y/o conclusiones finales comparando las dos alternativas realizadas en el ejercicio.

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Cargamos el dataset específico de cine
imdb_ds = load_dataset("imdb")

# Cargamos el tokenizador de DistilBERT
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# Función para procesar los textos
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Tokenizamos todo el dataset
tokenized_imdb = imdb_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
from transformers import AutoModelForSequenceClassification

# Definimos el mapeo de etiquetas
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 1. Instalamos la librería faltante
%pip install evaluate

from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

# 2. Cargamos la métrica de exactitud
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 3. Configuración del entrenamiento
args = TrainingArguments(
    output_dir="distilbert-imdb",
    eval_strategy="epoch",  # Se evalúa al final de cada época
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,    # Suficiente para el ejercicio
    weight_decay=0.01,
    report_to="none"       # Evita errores con wandb si no estás logueado
)

# 4. Inicializamos el Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    compute_metrics=compute_metrics,
)

# 5. ¡A entrenar!
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
